[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/10_Deployment/04_Web_Deployment_ONNX_JS/Web_Deployment_Deep_Dive.ipynb)

# Web Deployment with ONNX Runtime Web — Deep Dive

A comprehensive treatment of browser-based ML inference: WebAssembly memory model,
WebGPU compute shaders, model loading optimization, and production deployment patterns.

---

## Table of Contents

| # | Section | Key Topics |
|---|---------|------------|
| 1 | [Browser ML Architecture](#1) | ORT Web internals, backend selection |
| 2 | [WebAssembly Memory Model](#2) | Linear memory, SIMD, threading |
| 3 | [WebGPU Compute Pipeline](#3) | Shader dispatch, buffer management |
| 4 | [Model Loading Optimization](#4) | Fetch strategies, caching, streaming |
| 5 | [Preprocessing Pipeline](#5) | Canvas → Tensor, normalization parity |
| 6 | [Web Workers and Concurrency](#6) | Off-main-thread inference |
| 7 | [Performance Profiling](#7) | Chrome DevTools, memory budgets |
| 8 | [Production Patterns](#8) | CDN, versioning, fallback chains |
| 9 | [Security and Privacy](#9) | Client-side inference benefits |
| 10 | [Framework Integration](#10) | React, Next.js, Web Components |

---

<a id='1'></a>
## 1. Browser ML Architecture

ONNX Runtime Web brings native-quality inference to the browser, with multiple execution backends optimized for different hardware capabilities.

### Architecture Overview

```
┌─────────────────────────────────────────────────────────────────────┐
│                         Browser Environment                          │
│                                                                       │
│  ┌─────────────────────────────────────────────────────────────────┐ │
│  │  JavaScript Application Layer                                   │ │
│  │  import * as ort from 'onnxruntime-web'                        │ │
│  └──────────────────────────┬──────────────────────────────────────┘ │
│                             │                                         │
│  ┌──────────────────────────▼──────────────────────────────────────┐ │
│  │  ONNX Runtime Web Core                                          │ │
│  │  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐      │ │
│  │  │  Graph   │  │  Memory  │  │  Session │  │  Tensor  │      │ │
│  │  │  Optim.  │  │  Planner │  │  Manager │  │  Alloc.  │      │ │
│  │  └──────────┘  └──────────┘  └──────────┘  └──────────┘      │ │
│  └──────────────────────────┬──────────────────────────────────────┘ │
│                             │                                         │
│  ┌──────────────────────────▼──────────────────────────────────────┐ │
│  │  Execution Backends                                             │ │
│  │  ┌───────────┐  ┌───────────┐  ┌───────────┐                  │ │
│  │  │   WASM    │  │  WebGL    │  │  WebGPU   │                  │ │
│  │  │  (CPU)    │  │  (GPU)    │  │  (GPU)    │                  │ │
│  │  │           │  │  Legacy   │  │  Modern   │                  │ │
│  │  │  SIMD +   │  │  Frag.   │  │  Compute  │                  │ │
│  │  │  Threads  │  │  Shaders  │  │  Shaders  │                  │ │
│  │  └───────────┘  └───────────┘  └───────────┘                  │ │
│  └─────────────────────────────────────────────────────────────────┘ │
└─────────────────────────────────────────────────────────────────────┘
```

### Backend Selection Decision

| Backend | Browser Support | Best For | Limitations |
|---------|----------------|----------|-------------|
| **WASM** | Universal (99%+) | Small models, broad compat | CPU-only, slower for large models |
| **WebGL** | ~97% | Legacy GPU acceleration | Texture-based, precision limits |
| **WebGPU** | ~60% (growing) | Large models, GPU compute | Requires modern browser + GPU |

### Performance Hierarchy

For typical vision models (MobileNet-class):

$$T_{\text{WebGPU}} < T_{\text{WebGL}} < T_{\text{WASM}} \quad \text{(when GPU available)}$$

But for very small models (< 1M params):

$$T_{\text{WASM}} < T_{\text{WebGPU}} \quad \text{(GPU dispatch overhead dominates)}$$

The crossover point depends on model size and GPU capability:

$$B_{\text{crossover}} \approx \frac{T_{\text{GPU\_dispatch}}}{T_{\text{WASM\_per\_op}} - T_{\text{GPU\_per\_op}}}$$

<a id='2'></a>
## 2. WebAssembly Memory Model

WASM provides a sandboxed linear memory that ORT uses for tensor storage and computation.

### Linear Memory Architecture

```
WASM Linear Memory (grows in 64KB pages):
┌────────────────────────────────────────────────────────────────────┐
│ Page 0    │ Page 1    │ Page 2    │ ... │ Page N-1  │ Page N    │
│ (64 KB)   │ (64 KB)   │ (64 KB)   │     │ (64 KB)   │ (64 KB)   │
├───────────┴───────────┼───────────┴─────┴───────────┼───────────┤
│    ORT Runtime        │       Model Weights          │Activations│
│    Code + Stack       │       (loaded)               │  (arena)  │
│    (~2-4 MB)          │       (variable)             │ (dynamic) │
└───────────────────────┴──────────────────────────────┴───────────┘

Total addressable: up to 4 GB (32-bit) or 16 GB (memory64 proposal)
```

### Memory Budget Calculation

$$M_{\text{WASM}} = M_{\text{ORT\_runtime}} + M_{\text{model\_weights}} + M_{\text{activations}} + M_{\text{scratch}}$$

Browser memory limits (practical):

| Platform | Typical WASM Limit | Notes |
|----------|-------------------|-------|
| Desktop Chrome | ~4 GB | Per-tab limit |
| Mobile Chrome | ~512 MB - 1 GB | Depends on device RAM |
| Safari (iOS) | ~512 MB | More aggressive limits |
| Firefox | ~4 GB | Similar to Chrome |

### WASM SIMD

WASM SIMD provides 128-bit vector operations (like ARM NEON / x86 SSE):

$$\text{Throughput}_{\text{SIMD}} = \frac{128}{\text{sizeof}(\text{dtype}) \times 8} \times \text{Throughput}_{\text{scalar}}$$

For FP32: 4× speedup; for INT8: 16× speedup (theoretical).

### WASM Multi-Threading

WASM threads use SharedArrayBuffer, requiring specific HTTP headers:

```
Cross-Origin-Opener-Policy: same-origin
Cross-Origin-Embedder-Policy: require-corp
```

Thread scaling on WASM:

$$T_{\text{inference}}(N) = \frac{T_{\text{inference}}(1)}{\min(N, N_{\text{effective}})} + T_{\text{overhead}}(N)$$

where $N_{\text{effective}}$ is limited by the model's parallelism and browser thread pool.

<a id='3'></a>
## 3. WebGPU Compute Pipeline

WebGPU exposes modern GPU compute capabilities to the browser, enabling efficient matrix operations through compute shaders.

### WebGPU Execution Flow

```
┌──────────────┐    ┌──────────────┐    ┌──────────────┐
│  JS/TS Code  │    │  Command     │    │  GPU         │
│              │    │  Encoder     │    │  Hardware    │
│  Create      │───▶│  Record      │───▶│  Execute     │
│  Buffers     │    │  Dispatches  │    │  Shaders     │
│  Upload Data │    │  Barriers    │    │  Read Back   │
└──────────────┘    └──────────────┘    └──────────────┘
       │                                        │
       └────────── Asynchronous ────────────────┘
```

### Compute Shader for MatMul

A simplified WebGPU compute shader for matrix multiplication:

```wgsl
@group(0) @binding(0) var<storage, read> A: array<f32>;
@group(0) @binding(1) var<storage, read> B: array<f32>;
@group(0) @binding(2) var<storage, read_write> C: array<f32>;

struct Params { M: u32, N: u32, K: u32 }
@group(0) @binding(3) var<uniform> params: Params;

@compute @workgroup_size(16, 16)
fn main(@builtin(global_invocation_id) gid: vec3<u32>) {
    let row = gid.x;
    let col = gid.y;
    if (row >= params.M || col >= params.N) { return; }
    
    var sum: f32 = 0.0;
    for (var k: u32 = 0u; k < params.K; k++) {
        sum += A[row * params.K + k] * B[k * params.N + col];
    }
    C[row * params.N + col] = sum;
}
```

### Performance Model

WebGPU performance for a matrix operation:

$$T_{\text{total}} = T_{\text{dispatch}} + T_{\text{compute}} + T_{\text{readback}}$$

$$T_{\text{compute}} = \frac{2MNK}{\text{GPU\_FLOPS} \times U_{\text{utilization}}}$$

$$T_{\text{readback}} = \frac{M \times N \times 4}{\text{PCIe\_bandwidth}}$$

The dispatch overhead $T_{\text{dispatch}} \approx 0.01\text{-}0.1$ ms is significant for small operations — this is why kernel fusion matters:

$$\text{Speedup}_{\text{fusion}} = \frac{N_{\text{ops}} \times T_{\text{dispatch}} + T_{\text{compute}}}{1 \times T_{\text{dispatch}} + T_{\text{compute\_fused}}}$$

### GPU Buffer Management

ORT-Web manages GPU buffers to minimize CPU↔GPU transfers:

```
┌─────────────┐                      ┌─────────────┐
│  CPU Memory │                      │  GPU Memory │
│             │    Upload (slow)     │             │
│  Input      │─────────────────────▶│  Input Buf  │
│  Tensor     │                      │             │
│             │                      │  Compute    │
│             │                      │  (fast)     │
│             │   Readback (slow)    │             │
│  Output     │◀─────────────────────│  Output Buf │
│  Tensor     │                      │             │
└─────────────┘                      └─────────────┘

Key: Keep intermediate tensors GPU-resident!
```

<a id='4'></a>
## 4. Model Loading Optimization

In web deployment, model loading is often the dominant latency — users wait seconds while the model downloads and compiles.

### Loading Latency Model

$$T_{\text{load}} = T_{\text{fetch}} + T_{\text{parse}} + T_{\text{compile}} + T_{\text{warmup}}$$

$$T_{\text{fetch}} = \frac{S_{\text{model}}}{B_{\text{network}}} + T_{\text{RTT}} \times N_{\text{chunks}}$$

For a 20 MB model on various connections:

| Connection | Bandwidth | Fetch Time | User Experience |
|-----------|-----------|------------|------------------|
| 3G | 1 Mbps | 160s | Unusable |
| 4G | 10 Mbps | 16s | Poor |
| WiFi | 50 Mbps | 3.2s | Acceptable |
| 5G/Fiber | 100+ Mbps | < 1.6s | Good |

### Optimization Strategies

**1. Model Quantization (reduce $S_{\text{model}}$):**

$$S_{\text{INT8}} = \frac{S_{\text{FP32}}}{4} \implies T_{\text{fetch}} \text{ reduced by 75\%}$$

**2. CDN with Edge Caching:**

$$T_{\text{fetch}}^{\text{cached}} = T_{\text{RTT}} \approx 5\text{-}20\text{ ms} \quad \text{(vs seconds for origin)}$$

**3. Progressive Loading:**

```
┌─────────────────────────────────────────────────────────┐
│  Progressive Model Loading Strategy                      │
│                                                          │
│  Phase 1: Critical path (head layers)                   │
│  ├── Fetch first 2MB ──── Show "loading" with progress   │
│  │                                                       │
│  Phase 2: Remaining weights                              │
│  ├── Stream rest in background ── Enable inference       │
│  │                                                       │
│  Phase 3: Optimization                                   │
│  └── Cache to IndexedDB ── Instant on next visit        │
└─────────────────────────────────────────────────────────┘
```

**4. IndexedDB Caching:**

$$T_{\text{load}}^{\text{repeat}} = T_{\text{IDB\_read}} + T_{\text{compile}} \approx 50\text{-}200\text{ ms}$$

vs. first load:

$$T_{\text{load}}^{\text{first}} = T_{\text{fetch}} + T_{\text{parse}} + T_{\text{compile}} \approx 2\text{-}10\text{ s}$$

### Cache Invalidation Strategy

```javascript
const MODEL_VERSION = 'v2.1.0';
const CACHE_KEY = `model-${MODEL_VERSION}`;

// Check cache → fetch if miss → store
async function loadModel() {
  const cached = await idb.get(CACHE_KEY);
  if (cached) return cached;
  const buffer = await fetch(`/models/model-${MODEL_VERSION}.onnx`);
  await idb.put(CACHE_KEY, buffer);
  return buffer;
}
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Model loading time analysis
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: Loading time vs model size at different bandwidths
model_sizes_mb = np.linspace(1, 100, 50)
bandwidths = {'3G (1 Mbps)': 1, '4G (10 Mbps)': 10, 'WiFi (50 Mbps)': 50, '5G (100 Mbps)': 100}

for label, bw in bandwidths.items():
    fetch_time = model_sizes_mb * 8 / bw  # seconds
    axes[0].plot(model_sizes_mb, fetch_time, linewidth=2, label=label)

axes[0].axhline(y=3, color='green', linestyle='--', alpha=0.7, label='3s (good UX)')
axes[0].axhline(y=10, color='red', linestyle='--', alpha=0.7, label='10s (abandon)')
axes[0].set_xlabel('Model Size (MB)')
axes[0].set_ylabel('Fetch Time (seconds)')
axes[0].set_title('Model Loading: $t = S_{model} / B_{network}$')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 30)

# Middle: Loading time breakdown
components = ['Fetch\n(network)', 'Parse\n(protobuf)', 'Compile\n(WASM/GPU)', 'Warmup\n(first run)']
first_load = [3.0, 0.3, 0.8, 0.2]  # seconds (20MB model, WiFi)
cached_load = [0.05, 0.2, 0.8, 0.2]  # from IndexedDB

x = np.arange(len(components))
width = 0.35
axes[1].bar(x - width/2, first_load, width, label='First Load', color='#e74c3c')
axes[1].bar(x + width/2, cached_load, width, label='Cached (IndexedDB)', color='#2ecc71')
axes[1].set_xticks(x)
axes[1].set_xticklabels(components)
axes[1].set_ylabel('Time (seconds)')
axes[1].set_title('Loading Time Breakdown (20MB Model)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Right: Backend initialization comparison
backends = ['WASM', 'WASM\n(threaded)', 'WebGL', 'WebGPU']
init_times = [0.5, 0.8, 1.2, 1.5]  # Session creation time (s)
inference_times = [45, 22, 18, 8]  # ms per inference
first_inference = [t_init * 1000 + t_inf for t_init, t_inf in zip(init_times, inference_times)]

ax2 = axes[2]
bars = ax2.bar(backends, inference_times, color=['#3498db', '#2980b9', '#f39c12', '#2ecc71'], alpha=0.8)
ax2.set_ylabel('Inference Latency (ms)')
ax2.set_title('Steady-State Inference by Backend\n(MobileNetV3, after init)')
ax2.grid(True, alpha=0.3)

for bar, init_t in zip(bars, init_times):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'init: {init_t:.1f}s', ha='center', fontsize=9, style='italic')

plt.tight_layout()
plt.savefig('web_loading_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("Loading optimization analysis:")
print(f"  First load (WiFi, 20MB): {sum(first_load):.1f}s")
print(f"  Cached load (IndexedDB): {sum(cached_load):.2f}s")
print(f"  Speedup from caching: {sum(first_load)/sum(cached_load):.1f}×")

<a id='5'></a>
## 5. Preprocessing Pipeline

Preprocessing parity between training and serving is the #1 correctness concern in web deployment. The browser provides different image APIs than Python/OpenCV.

### Canvas → Tensor Conversion

```
┌──────────┐    ┌──────────────┐    ┌──────────────┐    ┌───────────┐
│  <img>   │───▶│  <canvas>    │───▶│  ImageData   │───▶│  Tensor   │
│  element │    │  drawImage() │    │  RGBA uint8  │    │  NCHW f32 │
└──────────┘    └──────────────┘    └──────────────┘    └───────────┘
     │                │                    │                   │
     │ Load/decode    │ Resize             │ Extract RGB,      │ Normalize
     │ (async)        │ (bilinear)         │ HWC→CHW          │ (mean/std)
```

### Mathematical Preprocessing Steps

**1. Resize** (bilinear interpolation):

$$I_{\text{resized}}(x', y') = \sum_{i,j} I(i,j) \cdot \max(0, 1-|x'-x_i|) \cdot \max(0, 1-|y'-y_j|)$$

**2. Normalize** to [0, 1]:

$$x_{\text{norm}} = \frac{x_{\text{uint8}}}{255}$$

**3. Standardize** with ImageNet statistics:

$$x_{\text{std}} = \frac{x_{\text{norm}} - \mu}{\sigma} \quad \text{where } \mu = [0.485, 0.456, 0.406], \; \sigma = [0.229, 0.224, 0.225]$$

**4. Layout transform** HWC → NCHW:

$$T[n, c, h, w] = x_{\text{std}}[h, w, c] \quad \text{(transpose last 3 dims)}$$

### Critical Differences from Python/OpenCV

| Operation | Python (PIL/OpenCV) | Browser (Canvas) |
|-----------|--------------------|-----------------|
| Resize algorithm | Configurable (LANCZOS, BILINEAR) | Browser-dependent |
| Color space | RGB (PIL) / BGR (OpenCV) | RGBA |
| Alpha handling | Explicit | Pre-multiplied in some contexts |
| Subpixel sampling | Deterministic | May vary across browsers |

### Numerical Precision Concern

Canvas operations use 8-bit precision internally. For models sensitive to preprocessing:

$$\text{Error}_{\text{canvas}} = \frac{1}{255} \approx 0.004 \quad \text{per pixel per channel}$$

This propagates through normalization:

$$\text{Error}_{\text{normalized}} = \frac{1}{255 \cdot \sigma_c} \approx 0.017$$

Usually negligible for classification, but can matter for regression tasks.

In [ ]:
import numpy as np

# Demonstrate the preprocessing pipeline (Python equivalent of browser JS)

def browser_preprocess_simulation(image_hwc_uint8, target_size=224):
    """Simulate browser-side image preprocessing for ONNX inference.
    
    This Python code mirrors the JavaScript preprocessing that would run
    in the browser, ensuring parity between training and web inference.
    """
    H, W, C = image_hwc_uint8.shape
    assert C == 3 or C == 4, "Expected RGB or RGBA"
    
    # If RGBA, drop alpha channel
    if C == 4:
        image_hwc_uint8 = image_hwc_uint8[:, :, :3]
    
    # Step 1: Resize (nearest-neighbor for simplicity; real browser uses bilinear)
    if H != target_size or W != target_size:
        # Simple resize via indexing (production would use proper interpolation)
        row_indices = np.linspace(0, H-1, target_size).astype(int)
        col_indices = np.linspace(0, W-1, target_size).astype(int)
        image_hwc_uint8 = image_hwc_uint8[np.ix_(row_indices, col_indices)]
    
    # Step 2: Normalize to [0, 1]
    image_float = image_hwc_uint8.astype(np.float32) / 255.0
    
    # Step 3: ImageNet standardization
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    image_normalized = (image_float - mean) / std
    
    # Step 4: HWC → NCHW
    image_chw = np.transpose(image_normalized, (2, 0, 1))  # CHW
    image_nchw = image_chw[np.newaxis, ...]  # NCHW
    
    return image_nchw.astype(np.float32)


def softmax(logits):
    """Numerically stable softmax."""
    x = logits - np.max(logits)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x)


def top_k(probs, k=5):
    """Return top-k indices and probabilities."""
    indices = np.argsort(-probs)[:k]
    return indices, probs[indices]


# Simulate preprocessing a camera frame
fake_camera_frame = np.random.randint(0, 256, (480, 640, 4), dtype=np.uint8)  # RGBA

tensor = browser_preprocess_simulation(fake_camera_frame, target_size=224)
print(f"Input (camera frame): shape={fake_camera_frame.shape}, dtype={fake_camera_frame.dtype}")
print(f"Output (tensor): shape={tensor.shape}, dtype={tensor.dtype}")
print(f"  Channel 0 (R) - mean: {tensor[0,0].mean():.4f}, std: {tensor[0,0].std():.4f}")
print(f"  Channel 1 (G) - mean: {tensor[0,1].mean():.4f}, std: {tensor[0,1].std():.4f}")
print(f"  Channel 2 (B) - mean: {tensor[0,2].mean():.4f}, std: {tensor[0,2].std():.4f}")

# Simulate model output processing
fake_logits = np.random.randn(1000).astype(np.float32)
probs = softmax(fake_logits)
indices, values = top_k(probs)
print(f"\nTop-5 predictions:")
for i, (idx, prob) in enumerate(zip(indices, values)):
    print(f"  #{i+1}: class {idx} ({prob*100:.2f}%)")

<a id='6'></a>
## 6. Web Workers and Concurrency

**Rule #1 of web ML: Never block the main thread.**

Inference must run in a Web Worker to prevent UI jank. The browser's main thread has a 16ms budget (60fps) — any inference longer than this will cause visible stuttering.

### Worker Architecture

```
┌───────────────────────┐          ┌───────────────────────────────┐
│     Main Thread       │          │        Worker Thread           │
│                       │          │                                │
│  ┌──────────────┐    │  post    │  ┌──────────────────────┐     │
│  │  UI Render   │    │  Message │  │  ORT InferenceSession│     │
│  │  (60fps)     │    │ ──────▶  │  │  (loaded once)       │     │
│  └──────────────┘    │          │  └──────────┬───────────┘     │
│                       │          │             │                  │
│  ┌──────────────┐    │          │  ┌──────────▼───────────┐     │
│  │  Camera      │    │          │  │  Inference Loop      │     │
│  │  Capture     │    │          │  │  preprocess → run    │     │
│  └──────┬───────┘    │          │  └──────────┬───────────┘     │
│         │             │          │             │                  │
│         │ Transfer    │  result  │             │                  │
│         │ (zero-copy) │ ◀──────  │             │                  │
│         ▼             │          │             ▼                  │
│  ┌──────────────┐    │          │  Return output tensor         │
│  │  Display     │    │          │  (Transferable)               │
│  │  Results     │    │          │                                │
│  └──────────────┘    │          └───────────────────────────────┘
└───────────────────────┘
```

### Transferable Objects

Use `Transferable` to avoid copying large typed arrays:

```javascript
// Main thread: send image data to worker (zero-copy)
const imageData = ctx.getImageData(0, 0, 224, 224);
worker.postMessage({ pixels: imageData.data.buffer }, [imageData.data.buffer]);
```

Transfer cost:

$$T_{\text{transfer}} \approx 0 \quad \text{(ownership transfer, no copy)}$$

vs. structured clone:

$$T_{\text{clone}} = \frac{N_{\text{bytes}}}{\text{memcpy\_bandwidth}} \approx \frac{224 \times 224 \times 4}{10 \text{ GB/s}} \approx 0.02 \text{ ms}$$

Small for single images, but significant at 30fps or for large tensor arrays.

### Double-Buffering Pattern

For real-time camera inference, use double-buffering:

$$\text{FPS}_{\text{effective}} = \min\left(\text{FPS}_{\text{camera}}, \; \frac{1}{T_{\text{inference}}}\right)$$

With double-buffering, the camera captures the next frame while inference runs on the current frame — no frames are dropped if $T_{\text{inference}} < T_{\text{frame}}$.

<a id='7'></a>
## 7. Performance Profiling

### Chrome DevTools Performance Panel

Key metrics to track for web ML:

| Metric | Target | How to Measure |
|--------|--------|----------------|
| First Inference | < 3s (incl. load) | `performance.mark()` |
| Steady-State Latency | < 50ms | `performance.measure()` |
| Main Thread Blocking | 0ms | Long Task API |
| Memory (JS Heap) | < 200MB | `performance.memory` |
| GPU Memory | < 512MB | DevTools GPU panel |

### Memory Budget for Web ML

```
┌─────────────────────────────────────────────────────────┐
│  Browser Tab Memory Budget (~2 GB practical)             │
├─────────────────────────────────────────────────────────┤
│                                                          │
│  ┌────────────────┐  Browser overhead: ~100 MB          │
│  │ DOM + JS Heap  │  Application code: ~50 MB           │
│  │ (150 MB)       │                                     │
│  ├────────────────┤                                     │
│  │ WASM Linear    │  ORT runtime: ~10 MB                │
│  │ Memory         │  Model weights: varies              │
│  │ (model-dep.)   │  Activation arena: ~2× largest layer│
│  ├────────────────┤                                     │
│  │ GPU Buffers    │  Input/output buffers               │
│  │ (WebGPU only)  │  Intermediate compute buffers       │
│  └────────────────┘                                     │
└─────────────────────────────────────────────────────────┘
```

### Inference Timing Best Practices

```javascript
// Accurate timing (accounts for async GPU execution)
const start = performance.now();
const output = await session.run(feeds);
// For WebGPU: force GPU sync before timing
await output[outputName].getData();  // triggers readback
const elapsed = performance.now() - start;
```

Without the readback wait, WebGPU timing only measures dispatch time, not actual computation.

### Performance Monitoring in Production

$$\text{Web Vitals Impact} = \begin{cases}
\text{LCP affected} & \text{if model blocks above-fold content} \\
\text{INP affected} & \text{if inference blocks user interaction} \\
\text{CLS affected} & \text{if results cause layout shift}
\end{cases}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Web performance simulation and comparison
np.random.seed(42)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Backend comparison for different model sizes
model_params_M = [1, 3, 5, 10, 25, 50]
wasm_latency = [8, 22, 38, 75, 180, 400]  # ms
wasm_threaded = [5, 12, 20, 42, 100, 220]
webgl_latency = [15, 18, 22, 35, 65, 130]  # GPU dispatch overhead visible for small models
webgpu_latency = [12, 12, 15, 22, 45, 85]

axes[0, 0].plot(model_params_M, wasm_latency, 'o-', linewidth=2, label='WASM', color='#e74c3c')
axes[0, 0].plot(model_params_M, wasm_threaded, 's-', linewidth=2, label='WASM (threaded)', color='#f39c12')
axes[0, 0].plot(model_params_M, webgl_latency, '^-', linewidth=2, label='WebGL', color='#9b59b6')
axes[0, 0].plot(model_params_M, webgpu_latency, 'D-', linewidth=2, label='WebGPU', color='#2ecc71')
axes[0, 0].set_xlabel('Model Size (M params)')
axes[0, 0].set_ylabel('Inference Latency (ms)')
axes[0, 0].set_title('Backend Performance vs Model Size')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axhline(y=16, color='gray', linestyle=':', alpha=0.5)  # 60fps budget
axes[0, 0].text(1, 17, '60fps budget', fontsize=8, color='gray')

# Memory usage comparison
model_sizes = [5, 10, 20, 50]  # MB (FP32 weight file)
wasm_mem = [20, 30, 50, 110]  # Total WASM memory MB
webgpu_cpu_mem = [15, 20, 30, 60]  # CPU-side
webgpu_gpu_mem = [10, 18, 35, 85]  # GPU-side

x = np.arange(len(model_sizes))
width = 0.25
axes[0, 1].bar(x - width, wasm_mem, width, label='WASM (total)', color='#e74c3c')
axes[0, 1].bar(x, webgpu_cpu_mem, width, label='WebGPU (CPU)', color='#3498db')
axes[0, 1].bar(x + width, webgpu_gpu_mem, width, label='WebGPU (GPU)', color='#2ecc71')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels([f'{s}MB model' for s in model_sizes])
axes[0, 1].set_ylabel('Memory Usage (MB)')
axes[0, 1].set_title('Memory Footprint by Backend')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Thread scaling for WASM
threads = [1, 2, 4, 8]
# MobileNetV3 inference times at different thread counts
inference_by_threads = [45, 28, 20, 18]  # Diminishing returns
ideal = [45 / t for t in threads]

axes[1, 0].plot(threads, inference_by_threads, 'b-o', linewidth=2, markersize=8, label='Actual')
axes[1, 0].plot(threads, ideal, 'k--', linewidth=1.5, label='Ideal scaling')
axes[1, 0].set_xlabel('WASM Threads')
axes[1, 0].set_ylabel('Inference Time (ms)')
axes[1, 0].set_title('WASM Thread Scaling\n(MobileNetV3, requires COOP/COEP headers)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Browser compatibility matrix
browsers = ['Chrome', 'Firefox', 'Safari', 'Edge', 'Mobile Chrome', 'Mobile Safari']
features = ['WASM', 'SIMD', 'Threads', 'WebGL', 'WebGPU']
support = np.array([
    [1, 1, 1, 1, 1],  # Chrome
    [1, 1, 1, 1, 0.8],  # Firefox (WebGPU partial)
    [1, 1, 0.5, 1, 0.5],  # Safari (threads/WebGPU limited)
    [1, 1, 1, 1, 1],  # Edge
    [1, 1, 0.5, 1, 0.3],  # Mobile Chrome
    [1, 1, 0, 1, 0],  # Mobile Safari
])

im = axes[1, 1].imshow(support, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[1, 1].set_xticks(range(len(features)))
axes[1, 1].set_xticklabels(features)
axes[1, 1].set_yticks(range(len(browsers)))
axes[1, 1].set_yticklabels(browsers)
axes[1, 1].set_title('Browser Feature Support Matrix (2024)')
plt.colorbar(im, ax=axes[1, 1], label='Support Level')

for i in range(len(browsers)):
    for j in range(len(features)):
        text = '✓' if support[i, j] >= 0.8 else ('~' if support[i, j] >= 0.3 else '✗')
        axes[1, 1].text(j, i, text, ha='center', va='center', fontsize=12)

plt.tight_layout()
plt.savefig('web_performance_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

<a id='8'></a>
## 8. Production Patterns

### CDN and Model Serving

```
┌─────────────┐     ┌──────────────┐     ┌──────────────┐
│   Browser   │────▶│   CDN Edge   │────▶│   Origin     │
│             │     │   (cached)   │     │   (S3/GCS)   │
│  1. Check   │     │              │     │              │
│     IDB     │     │  model.onnx  │     │  model.onnx  │
│  2. Fetch   │     │  (gzipped)   │     │  (versioned) │
│     from CDN│     │              │     │              │
│  3. Cache   │     │  Cache-Ctrl: │     │              │
│     to IDB  │     │  max-age=∞   │     │              │
└─────────────┘     └──────────────┘     └──────────────┘
```

### Version Management

Use content-hash in filenames for cache-busting:

```
/models/classifier-abc123.onnx    ← immutable, cache forever
/models/manifest.json              ← short cache, points to current version
```

### Fallback Chain Pattern

```javascript
async function createSession(modelUrl) {
  // Try backends in order of preference
  const backends = ['webgpu', 'webgl', 'wasm'];
  
  for (const backend of backends) {
    try {
      const session = await ort.InferenceSession.create(modelUrl, {
        executionProviders: [backend]
      });
      console.log(`Using ${backend} backend`);
      return session;
    } catch (e) {
      console.warn(`${backend} failed:`, e.message);
    }
  }
  throw new Error('No backend available');
}
```

### Bundle Size Optimization

The `onnxruntime-web` package includes multiple backends:

| Import | Bundle Size | Use When |
|--------|-------------|----------|
| Full package | ~8 MB (WASM) + ~2 MB (JS) | Need all backends |
| WASM only | ~8 MB (WASM) + ~500 KB | WASM-only deployment |
| WebGPU only | ~500 KB (JS) | WebGPU-capable audience |

Tree-shaking with modern bundlers (webpack 5+, Vite) can eliminate unused code.

<a id='9'></a>
## 9. Security and Privacy

### Client-Side Inference Advantages

```
Traditional Cloud ML:                   Client-Side ML:
┌──────┐  raw data   ┌──────┐          ┌──────────────────────┐
│Client│────────────▶│Server│          │       Client          │
│      │  results    │      │          │  data ──▶ model ──▶ UI│
│      │◀────────────│      │          │  (never leaves device)│
└──────┘             └──────┘          └──────────────────────┘
  ⚠ Privacy risk:                       ✓ Privacy preserved:
  • Images sent to server               • No server round-trip
  • GDPR/CCPA compliance               • GDPR-friendly
  • Data breach risk                    • Zero data exposure
```

### Security Considerations

| Concern | Risk | Mitigation |
|---------|------|------------|
| Model theft | User can download .onnx | Watermarking, obfuscation, accept as cost |
| Model poisoning | Tampered model on CDN | Subresource integrity (SRI) hash |
| Side-channel attacks | Timing leaks info | Constant-time postprocessing |
| Adversarial inputs | Crafted inputs fool model | Input validation, confidence thresholds |

### Subresource Integrity

```html
<script src="model-loader.js" 
        integrity="sha384-oqVuAfXRKap7fdgcCY5uykM6+R9GqQ8K/uxy9rx7HNQlGYl1kPzQho1wx4JwY8w">
</script>
```

For the model file itself, verify hash after fetch:

```javascript
const buffer = await response.arrayBuffer();
const hash = await crypto.subtle.digest('SHA-256', buffer);
const expected = 'abc123...';
if (bufToHex(hash) !== expected) throw new Error('Model integrity check failed');
```

<a id='10'></a>
## 10. Framework Integration

### React Integration Pattern

```typescript
// useOnnxSession hook
function useOnnxSession(modelUrl: string) {
  const [session, setSession] = useState<ort.InferenceSession | null>(null);
  const [loading, setLoading] = useState(true);
  const [error, setError] = useState<Error | null>(null);
  
  useEffect(() => {
    let cancelled = false;
    async function load() {
      try {
        const sess = await ort.InferenceSession.create(modelUrl, {
          executionProviders: ['webgpu', 'wasm'],
        });
        if (!cancelled) {
          setSession(sess);
          setLoading(false);
        }
      } catch (e) {
        if (!cancelled) setError(e);
      }
    }
    load();
    return () => { cancelled = true; };
  }, [modelUrl]);
  
  return { session, loading, error };
}
```

### Next.js Considerations

- ORT Web is **client-only** — use dynamic imports with `ssr: false`
- Model files go in `/public` for static serving
- Configure webpack to handle `.wasm` files
- Set COOP/COEP headers in `next.config.js` for threading

### Lazy Loading Pattern

$$T_{\text{perceived}} = T_{\text{page\_load}} + T_{\text{user\_action}} + T_{\text{model\_load}}$$

By prefetching the model after page interactive:

$$T_{\text{perceived}} = T_{\text{page\_load}} + T_{\text{user\_action}} + \max(0, T_{\text{model\_load}} - T_{\text{user\_action}})$$

If user takes > 2s to interact (typical), model is already loaded → $T_{\text{perceived}} \approx T_{\text{page\_load}} + T_{\text{user\_action}}$

In [ ]:
import numpy as np
import onnxruntime as ort

# Demonstrate the Python equivalent of web preprocessing pipeline
# This ensures parity between Python-trained models and browser inference

print("=" * 60)
print("Web Deployment: Preprocessing Parity Verification")
print("=" * 60)

# ImageNet normalization constants (must match browser JS)
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def python_preprocess(image_uint8, size=224):
    """Reference preprocessing (Python/training side)."""
    # Resize (simplified)
    h, w = image_uint8.shape[:2]
    rows = np.linspace(0, h-1, size).astype(int)
    cols = np.linspace(0, w-1, size).astype(int)
    resized = image_uint8[np.ix_(rows, cols)]
    
    # To float [0, 1]
    normalized = resized.astype(np.float32) / 255.0
    
    # Standardize
    standardized = (normalized - IMAGENET_MEAN) / IMAGENET_STD
    
    # HWC → NCHW
    return np.transpose(standardized, (2, 0, 1))[np.newaxis]

def browser_preprocess(image_uint8, size=224):
    """Simulated browser preprocessing (matches JS implementation)."""
    # Canvas drawImage resize (bilinear approximated as nearest here)
    h, w = image_uint8.shape[:2]
    rows = np.linspace(0, h-1, size).astype(int)
    cols = np.linspace(0, w-1, size).astype(int)
    resized = image_uint8[np.ix_(rows, cols)]
    
    # getImageData returns RGBA uint8 → we take RGB
    rgb = resized[:, :, :3]  # Drop alpha if present
    
    # JS: pixel / 255.0
    normalized = rgb.astype(np.float32) / 255.0
    
    # JS: (pixel - mean) / std
    standardized = (normalized - IMAGENET_MEAN) / IMAGENET_STD
    
    # JS: transpose to NCHW using typed array indexing
    return np.transpose(standardized, (2, 0, 1))[np.newaxis]

# Generate test image
test_image = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)

# Compare outputs
python_tensor = python_preprocess(test_image)
browser_tensor = browser_preprocess(test_image)

max_diff = np.max(np.abs(python_tensor - browser_tensor))
print(f"\nPreprocessing parity check:")
print(f"  Python output shape: {python_tensor.shape}")
print(f"  Browser output shape: {browser_tensor.shape}")
print(f"  Max absolute difference: {max_diff:.10f}")
print(f"  Parity: {'PASS ✓' if max_diff < 1e-6 else 'FAIL ✗'}")

# ORT session info
print(f"\nONNX Runtime version: {ort.__version__}")
print(f"Available providers: {ort.get_available_providers()}")
print(f"\nWeb deployment checklist:")
print(f"  ☐ Preprocessing parity verified")
print(f"  ☐ Model cached in IndexedDB")
print(f"  ☐ Inference in Web Worker")
print(f"  ☐ Fallback backend configured")
print(f"  ☐ Loading progress shown to user")
print(f"  ☐ COOP/COEP headers set (for threading)")

## Summary

Web deployment brings ML directly to users without server infrastructure, but requires careful engineering around browser constraints.

### Core Equations

| Concept | Formula |
|---------|--------|
| **Loading time** | $t = S_{\text{model}} / B_{\text{network}}$ |
| **SIMD speedup** | $\text{Speedup} = 128 / (\text{bits\_per\_element})$ |
| **GPU crossover** | $B_{\text{cross}} = T_{\text{dispatch}} / (T_{\text{WASM}} - T_{\text{GPU}})$ |
| **Frame budget** | $T_{\text{inference}} < 16\text{ms}$ (60fps) |
| **Cache speedup** | $T_{\text{cached}} / T_{\text{first}} \approx 0.03$× |

### Key Takeaways

1. **Model loading dominates perceived latency** — cache aggressively with IndexedDB
2. **Backend selection is model-size dependent** — WebGPU wins for large models, WASM for small
3. **Never block the main thread** — Web Workers are mandatory for production
4. **Preprocessing parity is the #1 correctness concern** — validate browser vs training pipeline
5. **Progressive enhancement with fallbacks** — not all browsers support WebGPU/threads

---

*Next: [ONNX for NLP](../../11_Advanced_Topics_and_Projects/01_ONNX_for_NLP/ONNX_for_NLP_Deep_Dive.ipynb) explores transformer model export and optimization for natural language processing.*